# 03 — Khám phá và kiểm toán dữ liệu

Tương ứng src/eda.py. Đọc bảng trước, rồi tạo và xem từng nhóm hình thay vì chỉ in ra một JSON dài.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Snapshot giá FPT; nên chạy notebook 01 trước.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Xem vài dòng và thống kê

Không điền thiếu bằng dữ liệu tương lai. Thống kê mô tả giúp phát hiện đơn vị sai, ngày sai hoặc giá trị cực đoan.

In [ ]:
from features import load_price_data
prices = load_price_data(PRICE_PATH)
display(prices.head())
display(prices.tail())
display(prices.describe(include="all"))
display(prices.isna().sum().to_frame("Số ô thiếu"))

## 5. Chạy kiểm toán và lưu biểu đồ

JSON lưu ở reports/data_audit.json; hình ở figures/. Giữ các dòng nguồn bất nhất để có thể truy vết.

In [ ]:
from eda import run_eda
audit = run_with_log(run_eda, PRICE_PATH, FIGURES_DIR, REPORTS_DIR)
display(pd.Series({key: audit[key] for key in [
    "row_count", "date_start", "date_end", "duplicate_dates", "zero_volume_sessions", "ohlc_consistency_violations"
]}).to_frame("Kiểm toán"))
display(pd.Series(audit["next_session_direction_counts"]).to_frame("Số phiên"))

## 6. Giá và khối lượng theo thời gian

Giá tăng/giảm theo nhiều giai đoạn. Khối lượng là số cổ phiếu giao dịch; trục khối lượng dùng thang hiển thị nén để vẫn nhìn được phiên nhỏ.

In [ ]:
show_figure("01_fpt_close.png")
show_figure("02_fpt_volume.png")

## 7. Lợi nhuận ngày và số nhãn

Return = close hôm nay / close hôm trước − 1; 0,01 nghĩa là tăng 1%. Nhãn dùng close phiên kế tiếp so với phiên hiện tại, khác return vừa quan sát.

In [ ]:
show_figure("03_daily_return_distribution.png")
show_figure("04_next_session_class_balance.png")
display(pd.Series(audit["daily_return_quantiles"]).to_frame("Phân vị return"))

## 8. Kiểm tra các ngày OHLC bất nhất

high phải không thấp hơn open/close và low phải không cao hơn open/close. Ô này chỉ liệt kê để kiểm toán, không bịa giá thay thế.

In [ ]:
from eda import _ohlc_violations
display(_ohlc_violations(prices))
show_figure("00_pipeline_overview.png")